In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 36.5 MB/s eta 0:00:00


In [35]:
import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import gensim
from gensim.models import word2vec

from gensim.models import KeyedVectors #  implements word vectors
from gensim.test.utils import datapath, get_tmpfile
from gensim.scripts.glove2word2vec import glove2word2vec

from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.cm as cm

import spacy

from tqdm.auto import tqdm
tqdm.pandas()

import matplotlib.pyplot as plt
import re

from tqdm import tqdm
tqdm.pandas()
import datetime

### EDA

In [3]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USER')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!kaggle datasets download -d shantanudhakadd/email-spam-detection-dataset-classification

Dataset URL: https://www.kaggle.com/datasets/shantanudhakadd/email-spam-detection-dataset-classification
License(s): other
100% 211k/211k [00:00<00:00, 65.5MB/s]



In [4]:
!unzip -o email-spam-detection-dataset-classification.zip

Archive:  email-spam-detection-dataset-classification.zip
  inflating: spam.csv                


In [5]:
file_path = 'spam.csv'
df = pd.read_csv(file_path, encoding='latin-1')

df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [6]:
filtered_df = df[df['Unnamed: 2'].notnull()]
filtered_df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
95,spam,Your free ringtone is waiting to be collected....,PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
281,ham,\Wen u miss someone,the person is definitely special for u..... B...,why to miss them,"just Keep-in-touch\"" gdeve.."""
444,ham,\HEY HEY WERETHE MONKEESPEOPLE SAY WE MONKEYAR...,HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE ...,NaN,NaN
671,spam,SMS. ac sun0819 posts HELLO:\You seem cool,"wanted to say hi. HI!!!\"" Stop? Send STOP to ...",NaN,NaN
710,ham,Height of Confidence: All the Aeronautics prof...,"this wont even start........ Datz confidence..""",NaN,NaN


In [7]:
df.isnull().sum()

,0
v1,0
v2,0
Unnamed: 2,5522
Unnamed: 3,5560
Unnamed: 4,5566


In [8]:
df.duplicated().sum()

np.int64(403)

In [10]:
df = df.drop_duplicates().reset_index(drop = True)
df.shape

(5169, 5)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [11]:
df = df.rename(columns = {"v1": "label", "v2": "message"})
df = df[['label', 'message']]
df

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5164,spam,This is the 2nd time we have tried 2 contact u...
5165,ham,Will Ì_ b going to esplanade fr home?
5166,ham,"Pity, * was in mood for that. So...any other s..."
5167,ham,The guy did some bitching but I acted like i'd...


In [13]:
label_distribution = df['label'].value_counts(normalize=True)
label_distribution

,proportion
label,
ham,0.87367
spam,0.12633


Бачимо, що дані не збалансовані

###Preprocessing
Підготуємо допоміжні списки даних, які знадобляться нам для нормалізації.

In [14]:
#Перелік скорочень

contractions = {
"ain't": "am not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he's": "he is",
"how'd": "how did",
"how'll": "how will",
"how's": "how is",
"i'd": "i would",
"i'll": "i will",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'll": "it will",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"must've": "must have",
"mustn't": "must not",
"needn't": "need not",
"oughtn't": "ought not",
"shan't": "shall not",
"sha'n't": "shall not",
"she'd": "she would",
"she'll": "she will",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"that'd": "that would",
"that's": "that is",
"there'd": "there had",
"there's": "there is",
"they'd": "they would",
"they'll": "they will",
"they're": "they are",
"they've": "they have",
"wasn't": "was not",
"we'd": "we would",
"we'll": "we will",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"where'd": "where did",
"where's": "where is",
"who'll": "who will",
"who's": "who is",
"won't": "will not",
"wouldn't": "would not",
"you'd": "you would",
"you'll": "you will",
"you're": "you are"
}

In [16]:
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english')).union({'also', 'would', 'much', 'many'})

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [17]:
#Завантаження моделі spaCy en_core_web_sm для обробки англійського тексту, але відключає два компоненти:

#parser — компонент для розбір синтаксичної структури речень.
#ner (Named Entity Recognition) — компонент для розпізнавання іменованих сутностей (наприклад, імена, локації, дати тощо).

nlp = spacy.load("en_core_web_sm", disable = ['parser','ner'])

#Визначимо функцію нормалізації тексту.

def normalize_text(raw_review):

    # Remove html tags
    text = re.sub("<[^>]*>", " ", raw_review) # match <> and everything in between. [^>] - match everything except >

    # Remove emails
    text = re.sub("\S*@\S*[\s]+", " ", text) # match non-whitespace characters, @ and a whitespaces in the end

    # remove links
    text = re.sub("https?:\/\/.*?[\s]+", " ", text) # match http, s - zero or once, //,
                                                    # any char 0-unlimited, whitespaces in the end

     # Convert to lower case, split into individual words
    text = text.lower().split()

    # Replace contractions with their full versions
    text = [contractions.get(word) if word in contractions else word
            for word in text]

    # Re-splitting for the correct stop-words extraction
    text = " ".join(text).split()

    # Remove stop words
    text = [word for word in text if not word in stop_words]

    text = " ".join(text)

    # Remove non-letters
    text = re.sub("[^a-zA-Z' ]", "", text) # match everything except letters and '


    # Stem words. Need to define porter stemmer above
    # text = [stemmer.stem(word) for word in text.split()]

    # Lemmatize words. Need to define lemmatizer above
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if len(token.lemma_) > 1 ])

    # Remove excesive whitespaces
    text = re.sub("[\s]+", " ", text)

    # Join the words back into one string separated by space, and return the result.
    return(text)

<>:16: SyntaxWarning: invalid escape sequence '\S'
<>:19: SyntaxWarning: invalid escape sequence '\/'
<>:49: SyntaxWarning: invalid escape sequence '\s'
<>:16: SyntaxWarning: invalid escape sequence '\S'
<>:19: SyntaxWarning: invalid escape sequence '\/'
<>:49: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_4328/2723133120.py:16: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub("\S*@\S*[\s]+", " ", text) # match non-whitespace characters, @ and a whitespaces in the end
/tmp/ipykernel_4328/2723133120.py:19: SyntaxWarning: invalid escape sequence '\/'
  text = re.sub("https?:\/\/.*?[\s]+", " ", text) # match http, s - zero or once, //,
/tmp/ipykernel_4328/2723133120.py:49: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub("[\s]+", " ", text)


In [18]:
df['message'] = df['message'].apply(normalize_text)

In [21]:
df.head()

,label,message
0,ham,go jurong point crazy available bugis great wo...
1,ham,ok lar joking wif oni
2,spam,free entry wkly comp win fa cup final tkts st ...
3,ham,dun say early hor already say
4,ham,nah think go usf life around though


In [19]:
df.shape

(5169, 2)

### Training and Vqlidation sets

In [24]:
X_train, X_val, y_train, y_val =  train_test_split(
    df['message'], df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(X_train.shape , X_val.shape, y_train.shape, y_val.shape)

(4135,) (1034,) (4135,) (1034,)


 ### Перетворення текстових даних у числові вектори

In [26]:
bow_vectorizer = CountVectorizer(max_features=1000, ngram_range=(1, 2))
tfidf_vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))

# Fitting vectorizers on data
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_val_bow = bow_vectorizer.transform(X_val)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

print(X_train_bow.shape, X_val_bow.shape, X_train_tfidf.shape, X_val_tfidf.shape)

(4135, 1000) (1034, 1000) (4135, 1000) (1034, 1000)


### Використання попередньо навчених ембедингів

In [28]:
!kaggle datasets download -d leadbest/googlenewsvectorsnegative300

Dataset URL: https://www.kaggle.com/datasets/leadbest/googlenewsvectorsnegative300
License(s): other
100% 3.17G/3.17G [00:45<00:00, 75.2MB/s]



In [29]:
!unzip -o googlenewsvectorsnegative300.zip

Archive:  googlenewsvectorsnegative300.zip
  inflating: GoogleNews-vectors-negative300.bin  
  inflating: GoogleNews-vectors-negative300.bin.gz  


In [30]:
word2vec_path = 'GoogleNews-vectors-negative300.bin'
word2vec_embeddings = KeyedVectors.load_word2vec_format(word2vec_path, binary=True)

def text_to_embedding(text, embeddings, embedding_dim=300):
    words = text.split()
    vectors = [embeddings[word] for word in words if word in embeddings]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(embedding_dim)

embedding_dim = 300  # Google News Word2Vec це стандартна розмірність для цієї конкретної моделі Word2Vec
X_train_embed = np.array([text_to_embedding(text, word2vec_embeddings, embedding_dim) for text in X_train])
X_val_embed = np.array([text_to_embedding(text, word2vec_embeddings, embedding_dim) for text in X_val])

### Побудова та начання моделей

In [32]:
log_reg_bow = LogisticRegression(max_iter=1000, random_state=42, penalty = 'l2')
rf_bow = RandomForestClassifier(n_estimators=100, random_state=42)

log_reg_tfidf = LogisticRegression(max_iter=1000, random_state=42, penalty = 'l2')
rf_tfidf = RandomForestClassifier(n_estimators=100, random_state=42)

log_reg_embed = LogisticRegression(max_iter=1000, random_state=42, penalty = 'l2')
rf_embed = RandomForestClassifier(n_estimators=100, random_state=42)

In [33]:
models = {
    "BoW + Logistic Regression": (log_reg_bow, X_train_bow, X_val_bow),
    "BoW + Random Forest": (rf_bow, X_train_bow, X_val_bow),
    "TF-IDF + Logistic Regression": (log_reg_tfidf, X_train_tfidf, X_val_tfidf),
    "TF-IDF + Random Forest": (rf_tfidf, X_train_tfidf, X_val_tfidf),
    "Word2Vec + Logistic Regression": (log_reg_embed, X_train_embed, X_val_embed),
    "Word2Vec + Random Forest": (rf_embed, X_train_embed, X_val_embed),
}

# Навчання всіх моделей
for name, (model, X_tr, _) in models.items():
    model.fit(X_tr, y_train)
    print(f"Модель '{name}' успішно навчена.")

Модель 'BoW + Logistic Regression' успішно навчена.
Модель 'BoW + Random Forest' успішно навчена.
Модель 'TF-IDF + Logistic Regression' успішно навчена.
Модель 'TF-IDF + Random Forest' успішно навчена.
Модель 'Word2Vec + Logistic Regression' успішно навчена.
Модель 'Word2Vec + Random Forest' успішно навчена.


### Оцінка моделей

In [34]:
# мітки текстові ('ham', 'spam'), конвертуємо для ROC-AUC у 0 та 1
y_val_num = y_val.map({'ham': 0, 'spam': 1}) if y_val.dtype == 'object' else y_val
pos_label = 'spam' if 'spam' in y_val.values else 1

results = []

for name, (model, _, X_v) in models.items():
    y_pred = model.predict(X_v)

    # отримання ймовірностей для позитивного класу (спам)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_v)[:, 1]
    else:
        y_proba = model.decision_function(X_v)

    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, pos_label=pos_label, zero_division=0)
    rec = recall_score(y_val, y_pred, pos_label=pos_label, zero_division=0)
    f1 = f1_score(y_val, y_pred, pos_label=pos_label, zero_division=0)
    auc = roc_auc_score(y_val_num, y_proba)

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(auc, 4)
    })

df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))

| Model                          |   Accuracy |   Precision |   Recall |   F1-Score |   ROC-AUC |
|:-------------------------------|-----------:|------------:|---------:|-----------:|----------:|
| BoW + Logistic Regression      |     0.9778 |      0.9821 |   0.8397 |     0.9053 |    0.9889 |
| BoW + Random Forest            |     0.9768 |      0.9735 |   0.8397 |     0.9016 |    0.9878 |
| TF-IDF + Logistic Regression   |     0.9671 |      0.9899 |   0.7481 |     0.8522 |    0.9899 |
| TF-IDF + Random Forest         |     0.9797 |      0.9825 |   0.855  |     0.9143 |    0.9888 |
| Word2Vec + Logistic Regression |     0.9516 |      0.8649 |   0.7328 |     0.7934 |    0.9727 |
| Word2Vec + Random Forest       |     0.9555 |      0.9775 |   0.6641 |     0.7909 |    0.963  |


### ВИСНОВКИ

У ході роботи було порівняно різні способи векторизації тексту: Bag-of-Words, TF-IDF та Word2Vec, а також моделі Logistic Regression і Random Forest. Найкращий результат показала комбінація TF-IDF + Random Forest з F1-score = 0.9143, Accuracy = 0.9797 та ROC-AUC = 0.9888. TF-IDF і Bag-of-Words загалом працювали краще за Word2Vec, оскільки для визначення спаму важливими є конкретні слова та фрази, характерні для спам-повідомлень. Word2Vec показав нижчий результат, ймовірно, через усереднення векторів слів та відмінність між новинним корпусом Google News і текстами спам-повідомлень. Logistic Regression показала дуже високу Precision, але мала нижчий Recall, тобто пропускала більше спаму. Random Forest краще знаходив складні комбінації ознак і тому забезпечив вищий Recall. При цьому TF-IDF + Random Forest зберігала високу точність і мала невелику кількість хибних спрацьовувань. Отже, для цієї задачі TF-IDF виявився більш практичним методом представлення тексту, ніж усереднені Word2Vec-ембединги. Загалом результати показали, що для класифікації спаму прості частотні методи можуть бути ефективнішими за більш складні представлення тексту.